####Requirement
1. Load data from flight-time.json into a table
2. Table structure is given below

```
    FL_DATE DATE, 
    OP_CARRIER STRING, 
    OP_CARRIER_FL_NUM STRING, 
    ORIGIN STRING, 
    ORIGIN_CITY_NAME STRING, 
    DEST STRING, 
    DEST_CITY_NAME STRING, 
    CRS_DEP_TIME LONG, 
    DEP_TIME LONG, 
    WHEELS_ON INT, 
    TAXI_IN INT, 
    CRS_ARR_TIME LONG, 
    ARR_TIME LONG, 
    CANCELLED INT, 
    DISTANCE INT
```

####Read data from the flight-time.json file

In [0]:
flight_time_raw_df = (
    spark.read
        .format("json")
        .option("mode", "FAILFAST")
        .option("dateFormat", "M/d/yyyy")
        .load("/Volumes/dev/spark_db/datasets/spark_programming/data/flight-time.json")
)

#### Investigate the dataframe data and schema for problems

order of the source connector on json is mismatched with the order on the requirement of schema to be written to the table. Example: - the first column from the table reuqirement is FL_DATE with datatype as DATE. After investigation the spark flights dataframe holds first column as ARR_TIME and FL_DATE is somewhere down in the order with the datatype as 'STRING'

Potential Problems for failure when writing it to the table:
 1. Incorrect column order
 2. Incorrect schema 

In [0]:
flight_time_raw_df.limit(3).display()

ARR_TIME,CANCELLED,CRS_ARR_TIME,CRS_DEP_TIME,DEP_TIME,DEST,DEST_CITY_NAME,DISTANCE,FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,TAXI_IN,WHEELS_ON
1348,0,1400,1115,1113,ATL,"Atlanta, GA",946,1/1/2000,DL,1451,BOS,"Boston, MA",5,1343
1543,0,1559,1315,1311,ATL,"Atlanta, GA",946,1/1/2000,DL,1479,BOS,"Boston, MA",7,1536
1651,0,1721,1415,1414,ATL,"Atlanta, GA",946,1/1/2000,DL,1857,BOS,"Boston, MA",9,1642


The reason for that is because the JSON connector sorts the columns in alphabetical order and sometimes connectors cannot infer the schema correctly. 

To fix that we need to define the schema and infer the schema - this is called Schema on read

#### Define Dataframe schema before reading it

In [0]:
from pyspark.sql.types import StringType, LongType, IntegerType, DateType, StructType, StructField

flight_schema = StructType([
    StructField("FL_DATE", DateType()),
    StructField("OP_CARRIER", StringType()),
    StructField("OP_CARRIER_FL_NUM", StringType()),
    StructField("ORIGIN", StringType()),
    StructField("ORIGIN_CITY_NAME", StringType()),
    StructField("DEST", StringType()),
    StructField("DEST_CITY_NAME", StringType()),
    StructField("CRS_DEP_TIME", LongType()),
    StructField("DEP_TIME", LongType()),
    StructField("WHEELS_ON", IntegerType()),
    StructField("TAXI_IN", IntegerType()),
    StructField("CRS_ARR_TIME", LongType()),
    StructField("ARR_TIME", LongType()),
    StructField("CANCELLED", IntegerType()),
    StructField("DISTANCE", IntegerType())
])

#### Read datafile with schema-on-read

In [0]:
flight_time_raw_with_schema_df = (
    spark.read
        .format("json")
        .option("mode", "FAILFAST")
        .option("dateFormat", "M/d/yyyy")
        .schema(flight_schema)
        .load("/Volumes/dev/spark_db/datasets/spark_programming/data/flight-time.json")
)

#### Investigate the Dataframe data and schema for problems

In [0]:
flight_time_raw_with_schema_df.limit(3).display()

FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,DEST,DEST_CITY_NAME,CRS_DEP_TIME,DEP_TIME,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,CANCELLED,DISTANCE
2000-01-01,DL,1451,BOS,"Boston, MA",ATL,"Atlanta, GA",1115,1113,1343,5,1400,1348,0,946
2000-01-01,DL,1479,BOS,"Boston, MA",ATL,"Atlanta, GA",1315,1311,1536,7,1559,1543,0,946
2000-01-01,DL,1857,BOS,"Boston, MA",ATL,"Atlanta, GA",1415,1414,1642,9,1721,1651,0,946


#### Save the Dataframe to the table flight_time_raw

In [0]:
flight_time_raw_with_schema_df.write.mode("overwrite").saveAsTable("dev.spark_db.flight_time_raw")

In [0]:
%sql
select * from dev.spark_db.flight_time_raw limit 3

FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,DEST,DEST_CITY_NAME,CRS_DEP_TIME,DEP_TIME,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,CANCELLED,DISTANCE
2000-01-01,DL,1451,BOS,"Boston, MA",ATL,"Atlanta, GA",1115,1113,1343,5,1400,1348,0,946
2000-01-01,DL,1479,BOS,"Boston, MA",ATL,"Atlanta, GA",1315,1311,1536,7,1559,1543,0,946
2000-01-01,DL,1857,BOS,"Boston, MA",ATL,"Atlanta, GA",1415,1414,1642,9,1721,1651,0,946
